In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model


load_dotenv()
model=init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
)
#定义工具

In [3]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel, Field
from langchain_core.tools import tool
class WeatherSchema(BaseModel):
    city:str=Field(default="北京",description="具体城市名称")
    if_forecast:bool=Field(default=False,description="是否包含明日天气")

@tool("get_weather_and_forecast",description="当日天气可以包含明天的天气预报",
      args_schema=WeatherSchema)
def get_weather(city:str,if_forecast:bool):
    res = f"{city}今天天气不错"
    if if_forecast:
        res +="\n明天下雨"
    return res
print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '当日天气可以包含明天的天气预报', 'parameters': {'properties': {'city': {'default': '北京', 'description': '具体城市名称', 'type': 'string'}, 'if_forecast': {'default': False, 'description': '是否包含明日天气', 'type': 'boolean'}}, 'type': 'object'}}}


In [5]:
from langchain_core.messages import HumanMessage

#1.将工具绑定到模型上
model_with_tools=model.bind_tools([get_weather])
#2维护一个消息列表
messages=[HumanMessage("今天杭州天气咋样,明天呢")]
response=model_with_tools.invoke(messages)
messages.append(response)
#4.获取响应中的tool_calls
tool_calls=response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"]=="get_weather_and_forecast":
        # 调用大模型
         tool_message=get_weather.invoke(tool_call)
         messages.append(tool_message)
final_response=model.invoke(messages)
messages.append(final_response)
#遍历一下消息列表
for msg in  messages:
    msg.pretty_print()

================================ Human Message =================================

今天杭州天气咋样,明天呢
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_00_gYvcKvTY0edDh17ZZZuH0605)
 Call ID: call_00_gYvcKvTY0edDh17ZZZuH0605
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州今天天气不错
明天下雨
================================== Ai Message ==================================

杭州今天天气不错；明天会下雨，出门记得带伞🌂。


In [6]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel, Field
from langchain_core.tools import tool

#docstring的使用
@tool("get_weather_and_forecast",parse_docstring=True)
def get_weather(city:str="北京",if_forecast:bool=False):
    """
    查询当前天气,可以包含明天的预测
    
    Args:
     city:城市名称
     if_forecast: 是否包含明天的天气 
    """
    res = f"{city}今天天气不错"
    if if_forecast:
        res +="\n明天下雨"
    return res
print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '查询当前天气,可以包含明天的预测', 'parameters': {'properties': {'city': {'default': '北京', 'description': '城市名称', 'type': 'string'}, 'if_forecast': {'default': False, 'description': '是否包含明天的天气', 'type': 'boolean'}}, 'type': 'object'}}}


In [7]:
from langchain_core.messages import HumanMessage

#1.将工具绑定到模型上
model_with_tools=model.bind_tools([get_weather])
#2维护一个消息列表
messages=[HumanMessage("今天杭州天气咋样,明天呢")]
response=model_with_tools.invoke(messages)
messages.append(response)
#4.获取响应中的tool_calls
tool_calls=response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"]=="get _weather_and_forecast":
        # 调用大模型
         tool_message=get_weather.invoke(tool_call)
         messages.append(tool_message)
final_response=model.invoke(messages)
messages.append(final_response)
#遍历一下消息列表
for msg in  messages:
    msg.pretty_print()

================================ Human Message =================================

今天杭州天气咋样,明天呢
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_00_6w62A1QIe59jJMDn1hmz1771)
 Call ID: call_00_6w62A1QIe59jJMDn1hmz1771
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州今天天气不错
明天下雨
================================== Ai Message ==================================

杭州今天天气不错，明天会下雨。出门记得带伞！
